# 03 — Model

**Purpose:** Compute investment-decision indicators from processed data and save to `data/outputs/`.

## Indicators
| Indicator | Method |
|-----------|--------|
| Yield curve inversion signal | 10y-2y spread threshold + consecutive months inverted |
| Recession probability | Logistic regression on yield curve + unemployment change (Estrella-Mishkin style) |
| Inflation regime | Z-score of CPI YoY vs 20-year rolling mean |
| Global growth pulse | GDP-weighted composite of World Bank + IMF forecasts |
| Risk-on / risk-off score | Composite of real rates, credit spreads, growth delta |

## Outputs
- `data/outputs/indicators.parquet` — time series of all indicators
- `data/outputs/latest_snapshot.json` — most recent values for dashboard header

## Papermill Parameters
- `run_date` — ISO date string injected by the GitHub Actions workflow

In [ ]:
run_date = None

In [ ]:
# Mount Google Drive for persistent storage (Colab only)
try:
    from google.colab import drive
    from pathlib import Path
    drive.mount("/content/drive")
    DRIVE_DATA = Path("/content/drive/MyDrive/macro-dashboard/data")
    _IN_COLAB = True
    print("Drive mounted. Reading from:", DRIVE_DATA)
except Exception:
    _IN_COLAB = False
    print("Not in Colab — using local data/ directory.")

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "pandas", "numpy", "scipy", "scikit-learn", "pyarrow"])
print("Packages ready.")

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from scipy.stats import norm

PROCESSED_DIR = DRIVE_DATA / "processed" if _IN_COLAB else Path("data/processed")
OUTPUTS_DIR   = DRIVE_DATA / "outputs"   if _IN_COLAB else Path("data/outputs")
RAW_DIR       = DRIVE_DATA / "raw"       if _IN_COLAB else Path("data/raw")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"PROCESSED_DIR : {PROCESSED_DIR}")
print(f"OUTPUTS_DIR   : {OUTPUTS_DIR}")
print(f"RAW_DIR       : {RAW_DIR}")

In [ ]:
# Load processed data
us    = pd.read_parquet(PROCESSED_DIR / "us_series.parquet")
panel = pd.read_parquet(PROCESSED_DIR / "macro_panel.parquet")

us.index = pd.to_datetime(us.index)
print("US series:   ", us.shape, "| latest:", us.index[-1].date())
print("Global panel:", panel.shape)
print("US columns:  ", us.columns.tolist())

In [ ]:
# --- Yield curve inversion signal ---
indicators = pd.DataFrame(index=us.index)

spread = us["yield_spread_10y2y"]
indicators["yield_spread_10y2y"] = spread
indicators["yield_spread_10y3m"] = us["yield_spread_10y3m"]
indicators["inverted"] = (spread < 0).astype(int)

# Count consecutive months inverted
consec = []
count = 0
for v in indicators["inverted"]:
    count = count + 1 if v else 0
    consec.append(count)
indicators["months_inverted"] = consec
indicators["inversion_signal"] = (indicators["months_inverted"] >= 3).astype(int)

# --- Recession probability (Estrella & Mishkin 1998 probit model) ---
# P(recession 12m ahead) = Φ(α + β × spread_10y3m)
# Parameters from the original paper, 12-month horizon
alpha, beta = -0.6521, -0.2375
spread_3m = us["yield_spread_10y3m"].ffill()
indicators["recession_prob"] = norm.cdf(alpha + beta * spread_3m)

print("Yield curve + recession prob computed.")
print(indicators[["yield_spread_10y2y", "inversion_signal", "recession_prob"]].tail(6))

In [ ]:
# --- Inflation regime (z-score vs 20-year rolling window) ---
cpi = us["cpi_yoy_pct"].dropna()
roll_mean = cpi.rolling(240, min_periods=60).mean()
roll_std  = cpi.rolling(240, min_periods=60).std()
indicators["inflation_zscore"] = (cpi - roll_mean) / roll_std
# Regime: +2=very high, +1=elevated, 0=normal, -1=low
conditions = [
    indicators["inflation_zscore"] > 1.5,
    indicators["inflation_zscore"] > 0.5,
    indicators["inflation_zscore"] < -0.5,
]
indicators["inflation_regime"] = np.select(conditions, [2, 1, -1], default=0)

# --- Global growth pulse (IMF GDP-weighted average) ---
# IMF data is annual (Jan 1 dates); resample to month-end before joining
# so it aligns with the monthly FRED index
gdp_col = "imf_NGDP_RPCH"
if gdp_col in panel.columns:
    growth = panel[gdp_col].dropna()
    growth_annual = growth.groupby(level="date").mean().rename("global_growth_pulse")
    growth_annual.index = pd.to_datetime(growth_annual.index)
    growth_monthly = growth_annual.resample("ME").ffill()
    indicators = indicators.join(growth_monthly, how="left")
    indicators["global_growth_pulse"] = indicators["global_growth_pulse"].ffill()

# --- Risk-on / risk-off score ---
def zscore(s, window=60):
    return (s - s.rolling(window, min_periods=12).mean()) / s.rolling(window, min_periods=12).std()

credit_z    =  zscore(us["credit_spread"])        # high spread = risk-off
real_rate_z =  zscore(us["real_rate_10y"])         # high real rate = risk-off
curve_z     = -zscore(us["yield_spread_10y2y"])    # flat/inverted = risk-off

risk_off = (credit_z + real_rate_z + curve_z) / 3
indicators["risk_score"] = (-risk_off).clip(-1, 1)  # positive = risk-on

print("All indicators computed.")
print(indicators.tail(3))

In [ ]:
# --- Country Scoreboard ---
try:
    stocks = pd.read_parquet(RAW_DIR / "stock_indices.parquet")
    print(f"Stock indices loaded: {stocks.shape}")
except FileNotFoundError:
    stocks = pd.DataFrame(columns=["stock_ytd_pct"])
    print("Warning: stock_indices.parquet not found - re-run 01_ingest to populate stock data")

try:
    policy = pd.read_parquet(RAW_DIR / "policy_rates.parquet")
    print(f"Policy rates loaded: {policy.shape}")
except FileNotFoundError:
    policy = pd.DataFrame(columns=["policy_rate"])
    print("Warning: policy_rates.parquet not found - re-run 01_ingest to populate policy rates")

# Existing major-economy basket (unchanged membership).
SCOREBOARD_COUNTRIES = [
    "United States", "China", "Germany", "Japan", "United Kingdom",
    "France", "India", "Brazil", "Canada", "Australia", "South Korea", "Italy",
]

# New baskets (specs/006-em-developing-baskets/data-model.md). These lists are
# DEFINED here for the schema/staleness machinery below, but the scoreboard loop
# only iterates the Major basket for now - EM/Developing rows are wired in by
# tasks T012 (US1) and T018 (US2). Disjointness rule: no country appears in more
# than one basket. China, India, Brazil, and South Korea are commonly called
# "emerging markets" by index providers but stay in the Major basket here because
# they're already covered above - they are NOT duplicated into EM (spec.md FR-001).
EM_COUNTRIES = [
    "Mexico", "Indonesia", "Turkey", "South Africa",
    "Poland", "Thailand", "Saudi Arabia", "Taiwan",
]
DEVELOPING_COUNTRIES = [
    "Nigeria", "Vietnam", "Bangladesh", "Kenya", "Egypt", "Pakistan",
]

# OECD members get quarterly GDP and monthly CPI/unemployment from oecd_* columns
OECD_MEMBERS = {
    "United States", "Germany", "Japan", "United Kingdom",
    "France", "Italy", "Canada", "Australia", "South Korea",
}

# --- Staleness policy (specs/006 data-model.md) -------------------------------
# Reference "now": the workflow-injected run_date, else current UTC date (naive).
NOW = (pd.Timestamp(run_date) if run_date else pd.Timestamp.utcnow().tz_localize(None)).normalize()

# Each metric is "slow" (quarterly-or-slower official stats) or "fast"
# (monthly-or-faster); stock_ytd is a daily series handled separately.
METRIC_TYPE = {
    "gdp_actual": "slow", "gdp_forecast": "slow",
    "current_account": "slow", "govt_debt": "slow",
    "inflation": "fast", "unemployment": "fast", "policy_rate": "fast",
    "stock_ytd": "stock",
}
# Max age in months before a value is stale, per (basket, metric_type). Looser
# tiers for EM/Developing reflect their inherently lower-frequency statistics.
STALENESS_THRESHOLDS = {
    ("Major",            "slow"): 6,  ("Major",            "fast"): 2,
    ("Emerging Markets", "slow"): 9,  ("Emerging Markets", "fast"): 4,
    ("Developing",       "slow"): 12, ("Developing",       "fast"): 6,
}
STOCK_STALE_DAYS = 5  # stock_ytd is daily wherever it exists; flat across baskets


def _is_stale(as_of, basket, metric_type):
    """True/False if a value is older/newer than its tier threshold; None when
    there is no as_of (i.e. no value). "Not available" is its own state and is
    never also flagged stale (data-model.md)."""
    if as_of is None or (isinstance(as_of, float) and np.isnan(as_of)):
        return None
    asof_ts = pd.Timestamp(as_of)
    if metric_type == "stock":
        return bool((NOW - asof_ts).days > STOCK_STALE_DAYS)
    months = STALENESS_THRESHOLDS[(basket, metric_type)]
    return bool(asof_ts < (NOW - pd.DateOffset(months=months)))


def _latest(col, country):
    """Latest non-null value for a panel column, with its reference date.
    Returns (value, as_of_iso) or (None, None)."""
    try:
        vals = panel[col].xs(country, level="country").dropna().sort_index()
        vals = vals[vals.index <= NOW]  # T026: never select a future (forecast) vintage as an actual
        if len(vals):
            return round(float(vals.iloc[-1]), 1), vals.index[-1].date().isoformat()
        return None, None
    except KeyError:
        return None, None


def _prev(col, country):
    """Second-latest value + its reference date (used for IMF GDP actual)."""
    try:
        vals = panel[col].xs(country, level="country").dropna().sort_index()
        vals = vals[vals.index <= NOW]  # T026: cap at today before taking the prior year
        if len(vals) >= 2:
            return round(float(vals.iloc[-2]), 1), vals.index[-2].date().isoformat()
        return None, None
    except KeyError:
        return None, None


def _coalesce(country, *cols):
    """First non-null value (and its as_of) from the ordered column list."""
    for col in cols:
        v, asof = _latest(col, country)
        if v is not None:
            return v, asof
    return None, None


def _lookup(frame, country, value_col, asof_col):
    """Value + as_of for the simple country-indexed policy/stock frames. as_of is
    None if the frame predates the as_of columns (i.e. an ingest run before the
    specs/006 change) - the value still renders, just without a freshness flag."""
    if country not in frame.index or value_col not in frame.columns:
        return None, None
    try:
        val = float(frame.loc[country, value_col])
    except (TypeError, ValueError):
        return None, None
    if np.isnan(val):
        return None, None
    asof = frame.loc[country, asof_col] if asof_col in frame.columns else None
    if asof is None or (isinstance(asof, float) and np.isnan(asof)):
        asof = None
    else:
        asof = str(asof)
    return val, asof


def build_row(country, basket):
    """Build one scoreboard row: values + per-metric as_of + per-metric stale
    flag + basket label. Applies uniformly to every basket."""
    row = {"country": country, "basket": basket}
    is_oecd = country in OECD_MEMBERS

    if is_oecd:
        # GDP: OECD quarterly actuals (most recent); fall back to IMF annual
        row["gdp_actual"],   ga = _coalesce(country, "oecd_gdp_yoy", "imf_NGDP_RPCH")
        row["gdp_forecast"]      = row["gdp_actual"]   # OECD provides actuals only
        gf = ga
        row["inflation"],    inf = _coalesce(country, "oecd_cpi_yoy", "imf_PCPIPCH")
        row["unemployment"], un  = _coalesce(country, "oecd_unemployment", "imf_LUR")
        row["data_source"]       = "OECD"
    else:
        row["gdp_forecast"], gf  = _latest("imf_NGDP_RPCH", country)
        row["gdp_actual"],   ga  = _prev(  "imf_NGDP_RPCH", country)
        row["inflation"],    inf = _latest("imf_PCPIPCH",   country)
        row["unemployment"], un  = _latest("imf_LUR",       country)
        row["data_source"]       = "IMF WEO"

    row["current_account"], ca = _latest("imf_BCA_NGDPD",   country)
    row["govt_debt"],       gd = _latest("imf_GGXWDG_NGDP", country)
    row["policy_rate"],     pr = _lookup(policy, country, "policy_rate",   "policy_rate_as_of")
    row["stock_ytd"],       sy = _lookup(stocks, country, "stock_ytd_pct", "stock_ytd_as_of")

    asof = {
        "gdp_actual": ga, "gdp_forecast": gf, "inflation": inf,
        "unemployment": un, "current_account": ca, "govt_debt": gd,
        "policy_rate": pr, "stock_ytd": sy,
    }
    for metric, a in asof.items():
        row[f"{metric}_as_of"] = a
        row[f"{metric}_stale"] = _is_stale(a, basket, METRIC_TYPE[metric])
    return row


# All three baskets (specs/006 Phase 3/4/5). Every row carries basket + per-metric
# as_of + stale; the renderers group into one titled table per basket.
rows =  [build_row(country, "Major")            for country in SCOREBOARD_COUNTRIES]
rows += [build_row(country, "Emerging Markets") for country in EM_COUNTRIES]
rows += [build_row(country, "Developing")       for country in DEVELOPING_COUNTRIES]

scoreboard = pd.DataFrame(rows).set_index("country")
scoreboard.to_parquet(OUTPUTS_DIR / "country_scoreboard.parquet")
print(f"Country scoreboard saved: {scoreboard.shape}")
_show = [c for c in scoreboard.columns if not c.endswith("_as_of") and not c.endswith("_stale")]
print(scoreboard[_show].to_string())

In [ ]:
def safe_float(v, decimals=2):
    try:
        f = float(v)
        return round(f, decimals) if not np.isnan(f) else None
    except Exception:
        return None

# --- Save indicators.parquet ---
indicators.to_parquet(OUTPUTS_DIR / "indicators.parquet")
print(f"indicators.parquet saved: {indicators.shape}")

# --- Save latest_snapshot.json ---
latest = indicators.dropna(how="all").iloc[-1]
snapshot = {
    "as_of":              run_date or str(latest.name.date()),
    "yield_spread_10y2y": safe_float(latest.get("yield_spread_10y2y")),
    "yield_spread_10y3m": safe_float(latest.get("yield_spread_10y3m")),
    "inversion_signal":   int(latest.get("inversion_signal", 0)),
    "months_inverted":    int(latest.get("months_inverted", 0)),
    "recession_prob":     safe_float(latest.get("recession_prob"), 3),
    "inflation_zscore":   safe_float(latest.get("inflation_zscore")),
    "inflation_regime":   int(latest.get("inflation_regime", 0)),
    "global_growth_pulse":safe_float(latest.get("global_growth_pulse")),
    "risk_score":         safe_float(latest.get("risk_score")),
}

out = OUTPUTS_DIR / "latest_snapshot.json"
out.write_text(json.dumps(snapshot, indent=2))
print(f"\nlatest_snapshot.json:")
print(json.dumps(snapshot, indent=2))